# 06 · Praproses, model, penyetelan, dan prediksi

**Bab 3 Subbab 3.8 dan 3.9.2–3.9.4.**

Untuk setiap ambang (0,5 utama; 0,25 dan 1,0 sensitivitas), periode uji, model, dan konfigurasi:

1. **Penyetelan**: setiap kandidat hiperparameter dilatih dan dinilai pada tiga blok validasi internal. Skor kandidat =
   rata-rata Macro-F1 tiga blok. Kandidat yang gagal di satu blok tidak memenuhi syarat.
2. **Pelatihan akhir**: kandidat terbaik dilatih ulang pada seluruh data latih luar, lalu memprediksi periode uji.
3. **Sensitivitas parameter tetap**: hiperparameter terpilih K1 dipakai ulang untuk K2 dan K3.
4. **Diagnostik kelas mayoritas**: prediksi konstan kelas terbanyak data latih (urutan seri BUY, HOLD, SELL).

Notebook ini **tidak menampilkan skor periode uji**; skor dibaca di notebook 07.

**Input:** `05_dataset.parquet`, `05_blok.csv`, `04_status_k3.txt`
**Output:** `06_prediksi.parquet`, `06_kandidat.parquet`, `06_kegagalan.csv`

In [1]:
import tomllib
from pathlib import Path

# Cari folder proyek (folder yang memuat configs/phase4-p0.toml) agar notebook bisa dijalankan dari mana saja.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/phase4-p0.toml").exists())
CFG = tomllib.loads((ROOT / "configs/phase4-p0.toml").read_text(encoding="utf-8"))
OUT = ROOT / "data/interim/phase4-p0"     # folder hasil; diabaikan Git karena memuat data vendor
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
import itertools
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Ubah untuk mencoba cepat, misalnya PERIODE_DIJALANKAN = ["2023-H1"] dan AMBANG_DIJALANKAN = [0.5].
# Run resmi: keduanya None.
PERIODE_DIJALANKAN = None
AMBANG_DIJALANKAN = None

df = pd.read_parquet(OUT / "05_dataset.parquet")
blok = pd.read_csv(OUT / "05_blok.csv", parse_dates=["awal", "akhir", "keputusan_pertama"])
STATUS_K3 = (OUT / "04_status_k3.txt").read_text().strip()

KELAS = CFG["label"]["class_order"]                 # kode kelas 0 = SELL, 1 = HOLD, 2 = BUY
K1 = CFG["fitur"]["k1"]
K2 = K1 + CFG["fitur"]["teknikal"]
K3 = K2 + [c for c in df.columns if c.startswith("macro_")]
KONFIGURASI = {"K1": K1, "K2": K2} | ({"K3": K3} if STATUS_K3 == "LAYAK" else {})
print("Konfigurasi:", {k: len(v) for k, v in KONFIGURASI.items()}, "· status K3:", STATUS_K3)

Konfigurasi: {'K1': 9, 'K2': 13, 'K3': 59} · status K3: LAYAK


## 1. Kandidat hiperparameter (dibekukan di Bab 3)

In [3]:
LR = CFG["model"]["regresi_logistik"]
XG = CFG["model"]["xgboost"]
KANDIDAT = {
    "regresi_logistik": [{"C": c} for c in LR["c_values"]],
    "xgboost": [{"max_depth": d, "n_estimators": n, "learning_rate": lr}
                for d, n, lr in itertools.product(XG["max_depth"], XG["n_estimators"], XG["learning_rate"])],
}


def urutan_seri(model, param):
    """Aturan seri Bab 3: C lebih kecil; kedalaman, iterasi, lalu learning rate lebih kecil."""
    if model == "regresi_logistik":
        return (param["C"],)
    return (param["max_depth"], param["n_estimators"], param["learning_rate"])


pd.DataFrame([{"model": m, **p} for m, daftar in KANDIDAT.items() for p in daftar])

,model,C,max_depth,n_estimators,learning_rate
0,regresi_logistik,0.01,NaN,NaN,NaN
1,regresi_logistik,0.10,NaN,NaN,NaN
2,regresi_logistik,1.00,NaN,NaN,NaN
3,regresi_logistik,10.00,NaN,NaN,NaN
4,xgboost,NaN,3.0,100.0,0.05
5,xgboost,NaN,3.0,100.0,0.10
6,xgboost,NaN,3.0,300.0,0.05
7,xgboost,NaN,3.0,300.0,0.10
8,xgboost,NaN,5.0,100.0,0.05
9,xgboost,NaN,5.0,100.0,0.10


## 2. Praproses yang hanya belajar dari data latih

1. Kolom yang seluruhnya kosong pada data latih dibuang dan dicatat.
2. Nilai kosong diisi **median data latih**, ditambah kolom indikator "nilai ini kosong".
3. Standardisasi (rata-rata dan simpangan baku data latih):
   - Regresi Logistik: selalu.
   - XGBoost: pada K3, karena kolom makro bersatuan berbeda dinormalisasi dari data latih (Bab 3 Subbab 3.7.3).
     Pohon keputusan tidak terpengaruh skala, jadi ini tidak mengubah cara pohon membelah.

Data uji tidak pernah ikut menghitung median, rata-rata, atau simpangan baku.

In [4]:
class Praproses:
    def __init__(self, skala):
        self.skala = skala

    def fit(self, latih):
        self.dipakai = [c for c in latih.columns if latih[c].notna().any()]
        self.dibuang = [c for c in latih.columns if c not in self.dipakai]
        self.median = latih[self.dipakai].median()
        self.indikator = [c for c in self.dipakai if latih[c].isna().any()]
        self.scaler = StandardScaler().fit(self._matriks(latih)) if self.skala else None
        return self

    def _matriks(self, d):
        nilai = d[self.dipakai].fillna(self.median).to_numpy(float)
        kosong = d[self.indikator].isna().to_numpy(float)
        return np.column_stack([nilai, kosong])

    def transform(self, d):
        x = self._matriks(d)
        if self.scaler is not None:
            x = self.scaler.transform(x)
        assert np.isfinite(x).all()
        return x

## 3. Latih satu model dan prediksi

- **Bobot kelas** $w_c = n / (3 n_c)$ dari data latih (Persamaan bobot kelas Bab 3). Untuk Regresi Logistik ini sama
  dengan `class_weight="balanced"`.
- Regresi Logistik: L2, `lbfgs`, maksimum 2.000 iterasi. Tidak konvergen = kandidat gagal.
- XGBoost: `multi:softprob`, subsample 0,8, colsample_bytree 0,8, λ = 1, seed 42, tanpa *early stopping*.

In [5]:
class GagalEvaluasi(Exception):
    pass


def latih_prediksi(model, param, latih, uji, kolom, kolom_label):
    y = latih[kolom_label].map({k: i for i, k in enumerate(KELAS)}).to_numpy()
    n_kelas = np.bincount(y, minlength=3)
    if (n_kelas == 0).any():
        raise GagalEvaluasi("data latih tidak memuat tiga kelas")
    prep = Praproses(skala=model == "regresi_logistik" or kolom is K3).fit(latih[kolom])
    x_latih, x_uji = prep.transform(latih[kolom]), prep.transform(uji[kolom])
    bobot = len(y) / (3 * n_kelas)

    if model == "regresi_logistik":
        m = LogisticRegression(C=param["C"], solver=LR["solver"], max_iter=LR["max_iter"],
                               class_weight={i: bobot[i] for i in range(3)})
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            try:
                m.fit(x_latih, y)
            except ConvergenceWarning as e:
                raise GagalEvaluasi(f"tidak konvergen: {e}") from e
    else:
        m = XGBClassifier(objective=XG["objective"], num_class=3, subsample=XG["subsample"],
                          colsample_bytree=XG["colsample_bytree"], reg_lambda=XG["reg_lambda"],
                          random_state=CFG["experiment"]["seed"], n_jobs=XG["n_jobs"], tree_method=XG["tree_method"],
                          eval_metric="mlogloss", **param)
        m.fit(x_latih, y, sample_weight=bobot[y])
    return np.array(KELAS)[m.predict(x_uji).astype(int)], prep.dibuang


def macro_f1(benar, prediksi):
    return f1_score(benar, prediksi, labels=KELAS, average="macro", zero_division=0)

## 4. Penyetelan dengan validasi internal

In [6]:
def potong(periode, nama_blok):
    r = blok[(blok["periode"] == periode) & (blok["blok"] == nama_blok)].iloc[0]
    latih = df[df["target_end_time"] < r["keputusan_pertama"]]
    uji = df[(df["decision_time"] >= r["awal"]) & (df["decision_time"] < r["akhir"]) & (df["target_end_time"] < r["akhir"])]
    return latih, uji


def setel(model, kolom, kolom_label, periode, catatan):
    hasil = []
    for param in KANDIDAT[model]:
        skor, alasan = [], None
        for nama_blok in ("validasi-1", "validasi-2", "validasi-3"):
            latih, valid = potong(periode, nama_blok)
            try:
                pred, _ = latih_prediksi(model, param, latih, valid, kolom, kolom_label)
                skor.append(macro_f1(valid[kolom_label], pred))
            except GagalEvaluasi as e:
                alasan = f"{nama_blok}: {e}"
                break
        rata = float(np.mean(skor)) if alasan is None else None
        catatan.append({**param, "model": model, "skor_validasi": skor, "rata_validasi": rata, "gagal": alasan})
        if rata is not None:
            hasil.append((rata, param))
    if not hasil:
        return None
    return min(hasil, key=lambda h: (-h[0], *urutan_seri(model, h[1])))[1]

## 5. Jalankan eksperimen

Run resmi memakai ketiga ambang dan ketujuh periode. Waktu kira-kira 20 menit di MacBook Apple M5 (XGBoost
dijalankan satu utas agar hasilnya deterministik).

In [7]:
def mayoritas(latih, kolom_label):
    hitung = latih[kolom_label].value_counts()
    return min(CFG["label"]["majority_tie_order"], key=lambda k: (-hitung.get(k, 0), CFG["label"]["majority_tie_order"].index(k)))


AMBANG = [CFG["label"]["threshold_utama"], *CFG["label"]["threshold_sensitivitas"]]
prediksi, kandidat, kegagalan = [], [], []
mulai = time.time()

for ambang in AMBANG:
    if AMBANG_DIJALANKAN and ambang not in AMBANG_DIJALANKAN:
        continue
    kolom_label = "label_" + str(ambang).replace(".", "p")
    for periode in blok["periode"].unique():
        if PERIODE_DIJALANKAN and periode not in PERIODE_DIJALANKAN:
            continue
        latih, uji = potong(periode, "uji")
        dasar = {"ambang": ambang, "periode": periode, "time": uji["time"].to_numpy(), "benar": uji[kolom_label].to_numpy()}
        kelas_mayor = mayoritas(latih, kolom_label)
        prediksi.append(pd.DataFrame({**dasar, "model": "mayoritas", "konfigurasi": "-", "analisis": "diagnostik",
                                      "prediksi": kelas_mayor}))
        for model in KANDIDAT:
            param_k1 = None
            for konfig, kolom in KONFIGURASI.items():
                catatan = []
                param = setel(model, kolom, kolom_label, periode, catatan)
                kandidat += [{**c, "ambang": ambang, "periode": periode, "konfigurasi": konfig} for c in catatan]
                if param is None:
                    kegagalan.append({"ambang": ambang, "periode": periode, "model": model, "konfigurasi": konfig,
                                      "alasan": "semua kandidat gagal pada validasi internal"})
                    continue
                try:
                    pred, dibuang = latih_prediksi(model, param, latih, uji, kolom, kolom_label)
                except GagalEvaluasi as e:
                    kegagalan.append({"ambang": ambang, "periode": periode, "model": model, "konfigurasi": konfig, "alasan": str(e)})
                    continue
                info = {"model": model, "konfigurasi": konfig, "parameter": str(param), "kolom_dibuang": len(dibuang)}
                prediksi.append(pd.DataFrame({**dasar, **info, "analisis": "disetel", "prediksi": pred}))
                if konfig == "K1":
                    param_k1 = param
                elif param_k1 is not None:
                    pred_tetap, _ = latih_prediksi(model, param_k1, latih, uji, kolom, kolom_label)
                    prediksi.append(pd.DataFrame({**dasar, **info, "parameter": str(param_k1),
                                                  "analisis": "parameter_tetap_k1", "prediksi": pred_tetap}))
                print(f"{ambang:<5} {periode:13} {model:17} {konfig}  terpilih {param}  ({time.time() - mulai:.0f} dtk)")

0.5   2023-H1       regresi_logistik  K1  terpilih {'C': 0.01}  (0 dtk)


0.5   2023-H1       regresi_logistik  K2  terpilih {'C': 0.1}  (1 dtk)


0.5   2023-H1       regresi_logistik  K3  terpilih {'C': 10.0}  (1 dtk)


0.5   2023-H1       xgboost           K1  terpilih {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  (7 dtk)


0.5   2023-H1       xgboost           K2  terpilih {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  (15 dtk)


0.5   2023-H1       xgboost           K3  terpilih {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  (30 dtk)


... (116 outputs left out)

1.0   2026-Jan-Jul  regresi_logistik  K3  terpilih {'C': 0.01}  (880 dtk)


1.0   2026-Jan-Jul  xgboost           K1  terpilih {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.05}  (891 dtk)


1.0   2026-Jan-Jul  xgboost           K2  terpilih {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.1}  (906 dtk)


1.0   2026-Jan-Jul  xgboost           K3  terpilih {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.1}  (938 dtk)


In [8]:
prediksi = pd.concat(prediksi, ignore_index=True)
kandidat = pd.DataFrame(kandidat)
kandidat["skor_validasi"] = kandidat["skor_validasi"].map(lambda s: ";".join(f"{v:.12f}" for v in s))
prediksi.to_parquet(OUT / "06_prediksi.parquet", index=False)
kandidat.to_parquet(OUT / "06_kandidat.parquet", index=False)
pd.DataFrame(kegagalan, columns=["ambang", "periode", "model", "konfigurasi", "alasan"]).to_csv(OUT / "06_kegagalan.csv", index=False)
print("Baris prediksi:", len(prediksi), "· kandidat dicatat:", len(kandidat), "· kegagalan:", len(kegagalan))
print("Kandidat gagal pada validasi internal:", int(kandidat["gagal"].notna().sum()))

Baris prediksi: 656469 · kandidat dicatat: 756 · kegagalan: 0
Kandidat gagal pada validasi internal: 0
